In [40]:
import pandas as pd
from datetime import datetime
from src.reverse_convertible import ReverseConvertible



In [76]:
import pandas as pd

portfolio = pd.DataFrame({
    "product_id": [
        "CH1483491150",
        "CH1449111066",
        "CH1461018793"
    ],
    
    "product_type": [
        "BRC",
        "MBRC",
        "MBRC"
    ],
    
    "type_style": [
        "European",
        "European",
        "European"
    ],
    
    "underlyings": [
        ["ALCON"],
        ["ABB", "HOLCIM", "NOVARTIS", "ROCHE"],
        ["ABB", "LONZA", "NESTLE"]
    ],
    
    "underlying_isins": [
        ["CH0432492467"],
        [
            "CH0012221716",
            "CH0012214059",
            "CH0012005267",
            "CH0012032048"
        ],
        [
            "CH0012221716",
            "CH0013841017",
            "CH0038863350"
        ]
    ],
    
    "currency": [
        "CHF",
        "CHF",
        "CHF"
    ],
    
    "position_units": [
        10,
        5,
        1
    ],
    
    "notional": [
        1000,
        1000,
        10000
    ],
    
    "cost_price": [
        1.00,
        0.98,
        1.00
    ],
    
    "initial_levels": [
        [59.72],
        [35.00, 70.00, 90.00, 250.00],
        [53.94, 555.20, 72.49]
    ],
    
    "current_spots": [
        [58.76],
        [34.00, 68.00, 92.00, 245.00],
        [53.94, 555.20, 72.49]   # replace with live levels
    ],
    
    # strike = initial fixing level
    "strike": [
        [59.72],
        [35.00, 70.00, 90.00, 250.00],
        [53.94, 555.20, 72.49]
    ],
    
    "barrier_pct": [
        0.70,
        0.70,
        0.70
    ],
    
    "coupon": [
        0.04,
        0.0675,
        0.0866
    ],
    
    "initial_fixing_date": [
        "2025-11-10",
        "2025-12-30",
        "2025-08-19"
    ],
    
    # choose one meaning only
    # here I use final fixing date, not repayment date
    "maturity_date": [
        "2026-11-17",
        "2026-12-28",
        "2026-08-19"
    ],
    
    "barrier_breached": [
        False,
        True,
        False
    ]
})

In [77]:
portfolio.to_csv("data/raw/portfolio.csv", index=False)
portfolio

,product_id,product_type,type_style,underlyings,underlying_isins,currency,position_units,notional,cost_price,initial_levels,current_spots,strike,barrier_pct,coupon,initial_fixing_date,maturity_date,barrier_breached
0,CH1483491150,BRC,European,[ALCON],[CH0432492467],CHF,10,1000,1.00,[59.72],[58.76],[59.72],0.7,0.0400,2025-11-10,2026-11-17,False
1,CH1449111066,MBRC,European,"[ABB, HOLCIM, NOVARTIS, ROCHE]","[CH0012221716, CH0012214059, CH0012005267, CH0...",CHF,5,1000,0.98,"[35.0, 70.0, 90.0, 250.0]","[34.0, 68.0, 92.0, 245.0]","[35.0, 70.0, 90.0, 250.0]",0.7,0.0675,2025-12-30,2026-12-28,True
2,CH1461018793,MBRC,European,"[ABB, LONZA, NESTLE]","[CH0012221716, CH0013841017, CH0038863350]",CHF,1,10000,1.00,"[53.94, 555.2, 72.49]","[53.94, 555.2, 72.49]","[53.94, 555.2, 72.49]",0.7,0.0866,2025-08-19,2026-08-19,False


In [86]:
from datetime import datetime
import numpy as np


class ReverseConvertible:

    def __init__(self, row, final_levels):
        self.row = row
        
        # Basic inputs
        self.notional = row["notional"]
        self.position_units = row["position_units"]
        self.cost_price = row["cost_price"]
        self.coupon = row["coupon"]
        self.barrier_pct = row["barrier_pct"]
        self.product_type = row["product_type"]
        self.type_style = row["type_style"]
        
        # Underlyings
        self.underlyings = row["underlyings"]
        self.initial_levels = row["initial_levels"]
        self.strike_levels = row["strike"]
        self.current_spots = row["current_spots"]

        if len(final_levels) != len(self.current_spots):
            raise ValueError("Scenario length must match number of underlyings.")

        # final_levels here are scenario shocks in %
        self.final_levels = [
            spot * (1 + level / 100)
            for spot, level in zip(self.current_spots, final_levels)
        ]
        
    def is_multi(self):
        return len(self.initial_levels) > 1
    
    def performances(self):
        """
        Final performance vs strike.
        Better for payoff/redemption logic.
        """
        return [
            final / strike
            for final, strike in zip(self.final_levels, self.strike_levels)
        ]
    
    def current_performances(self):
        """
        Current spot performance vs strike.
        """
        return [
            spot / strike
            for spot, strike in zip(self.current_spots, self.strike_levels)
        ]
    
    def performance(self):
        """
        Relevant payoff performance:
        - BRC: single underlying performance
        - MBRC: worst-of performance
        """
        if self.is_multi():
            return min(self.performances())
        return self.performances()[0]
    
    def barrier_breaches_final(self):
        """
        European barrier observation at final fixing.
        Barrier level = strike * barrier_pct
        """
        return [
            final <= strike
            for final, strike in zip(self.final_levels, self.strike_levels)
        ]
    
    def barrier_breached(self):
        if self.type_style.lower() != "european":
            raise NotImplementedError("Only European style implemented.")
        return any(self.barrier_breaches_final())
    
    def redemption(self):
        """
        If no barrier event: nominal redemption.
        If barrier event: economic value of delivery.
        """
        if not self.barrier_breached():
            return self.notional
        return self.notional * self.performance()
    
    def total_product_time(self):
        start = datetime.strptime(self.row["initial_fixing_date"], "%Y-%m-%d")
        maturity = datetime.strptime(self.row["maturity_date"], "%Y-%m-%d")
        return (maturity - start).days / 365

    def coupon_payment(self):
        T_total = self.total_product_time()
        return self.notional * self.coupon * T_total
    
    def payoff_per_unit(self):
        return self.redemption() + self.coupon_payment()
    
    def total_payoff(self):
        return self.payoff_per_unit() * self.position_units
    
    def total_cost(self):
        return self.position_units * self.notional * self.cost_price
    
    def pnl(self):
        return self.total_payoff() - self.total_cost()
    
    def return_pct(self):
        total_cost = self.total_cost()
        if total_cost == 0:
            return np.nan
        return self.pnl() / total_cost
    
    
    def return_pa(self):
        days = (datetime.strptime(self.row["maturity_date"], "%Y-%m-%d")
                - datetime.strptime(self.row["initial_fixing_date"], "%Y-%m-%d")).days
        if days <= 0:
            return np.nan
        return self.return_pct() * 360 / days
    
    def current_barrier_distances(self):
        """
        Current normalized distance to barrier:
        current/strike - barrier_pct
        """
        return [
            (spot / strike) - self.barrier_pct
            for spot, strike in zip(self.current_spots, self.strike_levels)
        ]
    
    def distance_to_barrier(self):
        distances = self.current_barrier_distances()
        if self.is_multi():
            return min(distances)
        return distances[0]
    
    def break_even(self):
        return 1 - self.coupon * self.total_product_time()
    
    def worst_underlying(self):
        if self.is_multi():
            idx = np.argmin(self.performances())
            return self.underlyings[idx]
        return self.underlyings[0]
    
    def summary(self):
        return {
            "product_type": self.product_type,
            "is_multi": self.is_multi(),
            "performance": self.performance()-1,
            "barrier_breached": self.barrier_breached(),
            "worst_underlying": self.worst_underlying(),
            "payoff_per_unit": self.payoff_per_unit(),
            "total_payoff": self.total_payoff(),
            "total_cost": self.total_cost(),
            "pnl": self.pnl(),
            "return_pct": self.return_pct(),
            "return_pa": self.return_pa(),
            "distance_to_barrier": self.distance_to_barrier(),
            "break_even": self.break_even()
        }

In [89]:
rc = ReverseConvertible(portfolio.iloc[1], [-10, 5, 0, -3])
print(rc.final_levels)

[30.6, 71.4, 92.0, 237.65]


In [88]:
def build_product_analytics(portfolio: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in portfolio.iterrows():
        rc = ReverseConvertible(row)
        s = rc.summary()

        rows.append({
            "product_id": row["product_id"],
            "product_type": row["product_type"],
            "type_style": row["type_style"],
            "currency": row["currency"],
            "position_units": row["position_units"],
            "notional": row["notional"],
            "coupon": row["coupon"],
            "underlyings": ", ".join(row["underlyings"]),
            "n_underlyings": len(row["underlyings"]),
            "initial_fixing_date": row["initial_fixing_date"],
            "maturity_date": row["maturity_date"],

            "performance": s["performance"],
            "barrier_breached": s["barrier_breached"],
            "worst_underlying": s["worst_underlying"],
            "payoff_per_unit": s["payoff_per_unit"],
            "total_payoff": s["total_payoff"],
            "total_cost": s["total_cost"],
            "pnl": s["pnl"],
            "return_pct": s["return_pct"],
            "return_pa": s["return_pa"],
            "distance_to_barrier": s["distance_to_barrier"],
            "break_even": s["break_even"]
        })

    df = pd.DataFrame(rows)
    return df


def portfolio_summary_table(product_df: pd.DataFrame) -> pd.DataFrame:
    total_cost = product_df["total_cost"].sum()
    total_payoff = product_df["total_payoff"].sum()
    total_pnl = product_df["pnl"].sum()

    summary = pd.DataFrame([{
        "n_products": len(product_df),
        "n_brc": (product_df["product_type"] == "BRC").sum(),
        "n_mbrc": (product_df["product_type"] == "MBRC").sum(),
        "total_cost": total_cost,
        "total_payoff": total_payoff,
        "total_pnl": total_pnl,
        "portfolio_return_pct": total_pnl / total_cost if total_cost != 0 else np.nan,
        "avg_return_pct": product_df["return_pct"].mean(),
        "weighted_return_pct": np.average(
            product_df["return_pct"],
            weights=product_df["total_cost"]
        ) if total_cost != 0 else np.nan,
        "worst_product_pnl": product_df["pnl"].min(),
        "best_product_pnl": product_df["pnl"].max(),
        "barrier_breached_count": product_df["barrier_breached"].sum(),
        "near_barrier_count": (product_df["distance_to_barrier"] <= 0.05).sum()
    }])

    return summary


def underlying_lookthrough(portfolio: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in portfolio.iterrows():
        rc = ReverseConvertible(row)

        product_cost = rc.total_cost()
        n = len(row["underlyings"])
        alloc_cost = product_cost

        current_perfs = rc.current_performances()
        final_perfs = rc.performances()
        barrier_distances = rc.current_barrier_distances()

        for i, underlying in enumerate(row["underlyings"]):
            rows.append({
                "product_id": row["product_id"],
                "underlying": underlying,
                "isin": row["underlying_isins"][i],
                "allocated_cost": alloc_cost,
                "current_performance": current_perfs[i] - 1,
                "scenario_performance": final_perfs[i] - 1,
                "distance_to_barrier": barrier_distances[i],
                "is_worst_underlying": underlying == rc.worst_underlying()
            })

    df = pd.DataFrame(rows)

    summary = (
        df.groupby(["underlying", "isin"], as_index=False)
        .agg(
            n_products=("product_id", "nunique"),
            allocated_cost=("allocated_cost", "sum"),
            avg_current_performance=("current_performance", "mean"),
            avg_scenario_performance=("scenario_performance", "mean"),
            min_distance_to_barrier=("distance_to_barrier", "min"),
            worst_of_count=("is_worst_underlying", "sum")
        )
        .sort_values("allocated_cost", ascending=False)
        .reset_index(drop=True)
    )

    total_alloc = summary["allocated_cost"].sum()
    summary["weight"] = summary["allocated_cost"] / total_alloc if total_alloc != 0 else np.nan

    return summary


def barrier_watchlist(product_df: pd.DataFrame, threshold: float = 0.05) -> pd.DataFrame:
    watch = product_df.copy()
    watch["near_barrier_flag"] = watch["distance_to_barrier"] <= threshold

    return watch.sort_values(
        ["near_barrier_flag", "distance_to_barrier", "pnl"],
        ascending=[False, True, True]
    ).reset_index(drop=True)


def maturity_profile(product_df: pd.DataFrame, today=None) -> pd.DataFrame:
    if today is None:
        today = datetime.today()

    df = product_df.copy()
    df["maturity_date_dt"] = pd.to_datetime(df["maturity_date"])
    df["days_to_maturity"] = (df["maturity_date_dt"] - pd.Timestamp(today)).dt.days

    bins = [-np.inf, 30, 90, 180, 365, 730, np.inf]
    labels = ["<=1M", "1-3M", "3-6M", "6-12M", "1-2Y", ">2Y"]
    df["maturity_bucket"] = pd.cut(df["days_to_maturity"], bins=bins, labels=labels)

    out = (
        df.groupby("maturity_bucket", as_index=False, observed=False)
        .agg(
            n_products=("product_id", "count"),
            total_cost=("total_cost", "sum"),
            total_payoff=("total_payoff", "sum"),
            total_pnl=("pnl", "sum")
        )
    )

    return out


def scenario_attribution(product_df: pd.DataFrame) -> pd.DataFrame:
    df = product_df.copy()
    total_pnl = df["pnl"].sum()

    df["pnl_contribution_pct"] = df["pnl"] / total_pnl if total_pnl != 0 else np.nan
    df["cost_weight"] = df["total_cost"] / df["total_cost"].sum() if df["total_cost"].sum() != 0 else np.nan

    return df[
        [
            "product_id",
            "product_type",
            "underlyings",
            "worst_underlying",
            "barrier_breached",
            "total_cost",
            "pnl",
            "pnl_contribution_pct",
            "return_pct",
            "distance_to_barrier"
        ]
    ].sort_values("pnl", ascending=True).reset_index(drop=True)


def run_portfolio_analysis(portfolio: pd.DataFrame):
    product_df = build_product_analytics(portfolio)
    portfolio_df = portfolio_summary_table(product_df)
    underlying_df = underlying_lookthrough(portfolio)
    barrier_df = barrier_watchlist(product_df)
    maturity_df = maturity_profile(product_df)
    attribution_df = scenario_attribution(product_df)

    return {
        "product_analytics": product_df,
        "portfolio_summary": portfolio_df,
        "underlying_lookthrough": underlying_df,
        "barrier_watchlist": barrier_df,
        "maturity_profile": maturity_df,
        "scenario_attribution": attribution_df
    }


# =========================================================
# RUN
# =========================================================

results = run_portfolio_analysis(portfolio)

print("\n=== PRODUCT ANALYTICS ===")
print(results["product_analytics"])

print("\n=== PORTFOLIO SUMMARY ===")
print(results["portfolio_summary"])

print("\n=== UNDERLYING LOOK-THROUGH ===")
print(results["underlying_lookthrough"])

print("\n=== BARRIER WATCHLIST ===")
print(results["barrier_watchlist"])

print("\n=== MATURITY PROFILE ===")
print(results["maturity_profile"])

print("\n=== SCENARIO ATTRIBUTION ===")
print(results["scenario_attribution"])

TypeError: __init__() missing 1 required positional argument: 'final_levels'

performances: [0.8571428571428571, 1.0714285714285714, 0.9444444444444444, 0.96]
worst_performance: 0.8571428571428571
worst_underlying_index: 0
payoff_per_unit: 924.2729941291585
total_payoff: 4621.364970645793
total_cost: 4900.0
pnl: -278.6350293542073
return_pct: -0.05686429170494026
return_pa: -0.0752895136488213
distance_to_barrier: [0.27142857142857146, 0.27142857142857146, 0.3222222222222222, 0.28]
break_even: 0.9325
